# 🚀 Multi-Label Classification: Ablation Study (Kaggle T4)
EfficientNet-B0 + CBAM + ASL — Full pipeline A/B/C/D/E/F

## ⚠️ KEY FIXES (so với master gốc):
1. **evaluate.py**: KHÔNG dùng `autocast` khi evaluate → fp32 sigmoid → mAP chính xác (~+19% vs buggy)
2. **exp_C config**: Bỏ `eps: 1e-5` (gây mất gradient signal) → dùng default `eps=1e-8`
3. **exp_E, exp_F configs**: Đảm bảo không có `eps` thừa, đúng backbone/CBAM setting
4. Clone từ branch `main` đã chứa tất cả fixes


## Cell 1: Install Dependencies

In [ ]:
!pip install timm pycocotools torchmetrics scikit-learn matplotlib seaborn pyyaml tqdm -q
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")


## Cell 2: Clone Repository từ GitHub

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/Thinh59/ECAAL.git'  # ← đổi nếu URL khác
REPO_DIR = Path('/kaggle/working/ECAAL')

if REPO_DIR.exists():
    print(f'Repo đã tồn tại — pulling latest...')
    os.system(f'git -C {REPO_DIR} pull')
else:
    print(f'Cloning {REPO_URL} ...')
    ret = os.system(f'git clone {REPO_URL} {REPO_DIR}')
    if ret != 0:
        raise RuntimeError('git clone thất bại! Kiểm tra REPO_URL.')

# Verify tất cả files cần thiết
required = [
    'src/train.py', 'src/losses.py', 'src/models.py',
    'src/dataset.py', 'src/evaluate.py', 'src/cbam.py', 'src/utils.py',
    'configs/exp_A_resnet_bce.yaml', 'configs/exp_B_resnet_asl.yaml',
    'configs/exp_C_efficientnet_cbam_asl.yaml', 'configs/exp_D_resnet_focal.yaml',
    'configs/exp_E_resnet_cbam_asl.yaml', 'configs/exp_F_efficientnet_asl.yaml',
]
all_ok = True
for rel in required:
    p = REPO_DIR / rel
    ok = p.exists()
    print(f"  {'OK' if ok else 'MISSING'} {rel}")
    if not ok:
        all_ok = False

if not all_ok:
    raise FileNotFoundError('Một số file thiếu — clone chưa đầy đủ!')
print('\nRepository OK!')


## Cell 3: Verify Fixes

In [ ]:
import sys, inspect
sys.path.insert(0, str(REPO_DIR / 'src'))

# ── FIX 1: Verify ASL implementation ──────────────────────────────────────
from losses import AsymmetricLoss
src = inspect.getsource(AsymmetricLoss.forward)
fix_ok = 'xs_pos_shifted = (xs_pos - self.clip).clamp(min=0)' in src
bug_ok = 'xs_neg_shifted = (xs_neg + self.clip)' not in src
print(f"  {'✅' if fix_ok else '❌'} ASL: correct probability shifting")
print(f"  {'✅' if bug_ok else '❌'} ASL: old bug absent")

# ── FIX 2: Verify evaluate.py không có autocast ───────────────────────────
from evaluate import evaluate_model
eval_src = inspect.getsource(evaluate_model)
no_autocast = 'autocast' not in eval_src
print(f"  {'✅' if no_autocast else '❌'} evaluate_model: NO autocast (fp32 evaluation)")

# ── FIX 3: Verify exp_C config không có eps=1e-5 ──────────────────────────
import yaml
cfg_c = yaml.safe_load(open(REPO_DIR / 'configs/exp_C_efficientnet_cbam_asl.yaml'))
has_eps = 'eps' in cfg_c.get('loss', {})
print(f"  {'✅' if not has_eps else '⚠️  WARNING'} exp_C config: {'no eps (correct)' if not has_eps else f'eps={cfg_c["loss"]["eps"]} present — should be removed!'}")

# ── Test ASL không NaN ────────────────────────────────────────────────────
import torch
asl = AsymmetricLoss(gamma_pos=0, gamma_neg=4, clip=0.05)
logits  = torch.tensor([[10., -10., 5., -5.]] * 8)
targets = torch.randint(0, 2, (8, 4)).float()
loss = asl(logits, targets)
print(f"  {'✅' if not torch.isnan(loss) else '❌'} ASL: no NaN on saturated logits (loss={loss.item():.4f})")

if fix_ok and bug_ok and no_autocast:
    print("\n🎉 ALL FIXES VERIFIED — Ready to train!")
else:
    raise RuntimeError("❌ Một số fix chưa được apply — kiểm tra lại repo!")


## Cell 4: Setup Directories

In [ ]:
os.makedirs('/kaggle/working/data/coco_subset', exist_ok=True)
os.makedirs('/kaggle/working/outputs', exist_ok=True)
os.chdir('/kaggle/working')
print(f"Working dir: {os.getcwd()}")
print("✅ Directory structure created")


## Cell 5: Dataset Preparation

**Trước khi chạy:** Notebook → **+ Add Data** → search `coco-2017-dataset` (awsaf49) → Add.
Dataset sẽ mount tại `/kaggle/input/coco-2017-dataset/`.


In [ ]:
# Verify COCO dataset
coco_root = '/kaggle/input/coco-2017-dataset'
if os.path.exists(coco_root):
    print("✅ COCO 2017 dataset found")
    for name, path in [
        ('train annotations', f'{coco_root}/annotations/instances_train2017.json'),
        ('val annotations',   f'{coco_root}/annotations/instances_val2017.json'),
        ('train2017/',        f'{coco_root}/train2017'),
        ('val2017/',          f'{coco_root}/val2017'),
    ]:
        print(f"  {'OK' if os.path.exists(path) else 'MISSING'} {name}")
else:
    print(f"❌ COCO dataset not found at {coco_root}")
    print("   → Notebook → + Add Data → coco-2017-dataset (awsaf49)")


## Cell 6: Create COCO 2017 Subset (16k train + 4k val)

In [ ]:
import json, random
import numpy as np
from collections import defaultdict
from pathlib import Path

SUBSET_DIR = Path('/kaggle/working/data/coco_subset')
train_f = SUBSET_DIR / 'subset_train_ids.json'
val_f   = SUBSET_DIR / 'subset_val_ids.json'

if train_f.exists() and val_f.exists():
    print(f"✅ Subset đã tồn tại — skip tạo lại")
    print(f"   train: {len(json.load(open(train_f)))} ids")
    print(f"   val:   {len(json.load(open(val_f)))} ids")
else:
    sys.path.insert(0, str(REPO_DIR / 'src'))
    from dataset import create_coco_subset
    create_coco_subset(
        coco_root=coco_root,
        output_dir=str(SUBSET_DIR),
        num_train=16000,
        num_val=4000,
        seed=42,
    )
    print("✅ Subset created!")


## Cell 7: Train Experiment A — ResNet50 + BCE (Baseline)

In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_A_resnet_bce.yaml


## Cell 8: Train Experiment B — ResNet50 + ASL

In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_B_resnet_asl.yaml


## Cell 9: Train Experiment D — ResNet50 + Focal Loss

In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_D_resnet_focal.yaml


## Cell 10: Train Experiment C — EfficientNet-B0 + CBAM + ASL ⭐ MAIN

**Expected mAP: ~0.75** (vs 0.56 trong master bị bug)

Fix so với master:
- `evaluate.py`: không dùng autocast → fp32 precision → mAP chính xác
- `exp_C config`: không có `eps=1e-5` → gradient signal đầy đủ


In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_C_efficientnet_cbam_asl.yaml


## Cell 11: Train Experiment E — ResNet50 + CBAM + ASL

Ablation: so sánh ResNet50 không CBAM (Exp B) vs có CBAM (Exp E)
Expected mAP: ~0.71-0.73


In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_E_resnet_cbam_asl.yaml


## Cell 12: Train Experiment F — EfficientNet-B0 + ASL (không CBAM)

Ablation: so sánh EffNet không CBAM (Exp F) vs có CBAM (Exp C)
Expected mAP: ~0.70-0.72


In [ ]:
!cd /kaggle/working && python /kaggle/working/ECAAL/src/train.py --config /kaggle/working/ECAAL/configs/exp_F_efficientnet_asl.yaml


## Cell 13: Evaluate Results & Plot Curves

In [ ]:
import json, os
import pandas as pd
import matplotlib.pyplot as plt

base = '/kaggle/working/outputs'
exps = {
    'A: ResNet50+BCE':       'exp_A_resnet_bce/log.json',
    'D: ResNet50+Focal':     'exp_D_resnet_focal/log.json',
    'B: ResNet50+ASL':       'exp_B_resnet_asl/log.json',
    'E: ResNet50+CBAM+ASL':  'exp_E_resnet_cbam_asl/log.json',
    'F: EffNet+ASL':         'exp_F_efficientnet_asl/log.json',
    'C: EffNet+CBAM+ASL':    'exp_C_efficientnet_cbam_asl/log.json',
}

rows = []
logs = {}
for name, rel in exps.items():
    path = os.path.join(base, rel)
    if not os.path.exists(path):
        print(f"⚠️  {name} log not found: {path}")
        continue
    records = json.load(open(path))
    logs[name] = records
    best = max(records, key=lambda r: r.get('mAP', 0))
    rows.append({
        'Experiment': name,
        'mAP': f"{best['mAP']:.4f}",
        'Macro F1': f"{best['macro_f1']:.4f}",
        'Best Epoch': best['epoch'],
    })

print("\n" + "="*65)
print("ABLATION STUDY RESULTS")
print("="*65)
print(pd.DataFrame(rows).to_string(index=False))
print("="*65)


## Cell 14: Plot Training Curves

In [ ]:
if logs:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    colors = ['#e74c3c', '#f39c12', '#3498db', '#9b59b6', '#1abc9c', '#2ecc71']

    for (name, recs), c in zip(logs.items(), colors):
        ep = [r['epoch'] for r in recs]
        ax1.plot(ep, [r['mAP'] for r in recs], marker='o', label=name, color=c, linewidth=2)
        ax2.plot(ep, [r['macro_f1'] for r in recs], marker='s', label=name, color=c, linewidth=2)

    for ax, title, ylabel in [(ax1, 'mAP vs Epoch', 'mAP'),
                               (ax2, 'Macro F1 vs Epoch', 'Macro F1')]:
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.legend(fontsize=8, loc='best')
        ax.grid(alpha=0.3, linestyle='--')

    plt.tight_layout()
    plt.savefig(os.path.join(base, 'ablation_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Curves saved")
else:
    print("⚠️  No log files found. Run training cells first.")


## Cell 15: Summary Statistics

In [ ]:
if rows:
    df = pd.DataFrame(rows)
    print("\n📊 ABLATION SUMMARY:")
    print(f"  Total Experiments: {len(rows)}")
    best_exp = max(rows, key=lambda x: float(x['mAP']))
    print(f"  Best mAP: {best_exp['Experiment']} = {best_exp['mAP']}")

    # CBAM contribution (Exp C vs Exp F)
    c_row = next((r for r in rows if r['Experiment'].startswith('C:')), None)
    f_row = next((r for r in rows if r['Experiment'].startswith('F:')), None)
    if c_row and f_row:
        delta = float(c_row['mAP']) - float(f_row['mAP'])
        print(f"\n  CBAM contribution (EffNet backbone): +{delta:.4f} mAP")
        print(f"    Exp F (no CBAM): {f_row['mAP']}  →  Exp C (CBAM): {c_row['mAP']}")

    # ASL vs BCE (Exp B vs Exp A)
    a_row = next((r for r in rows if r['Experiment'].startswith('A:')), None)
    b_row = next((r for r in rows if r['Experiment'].startswith('B:')), None)
    if a_row and b_row:
        delta = float(b_row['mAP']) - float(a_row['mAP'])
        print(f"\n  ASL contribution (ResNet50 backbone): +{delta:.4f} mAP")
        print(f"    Exp A (BCE): {a_row['mAP']}  →  Exp B (ASL): {b_row['mAP']}")

    print(f"\n  Logs saved to: {base}/*/log.json")
else:
    print("No experiments completed yet.")


## Cell 16: Cross-Dataset Evaluation (VOC 2012)
Đánh giá các model đã train trên tập test của Pascal VOC 2012 và nén kết quả lại.

In [ ]:
import os

# Script đánh giá chéo VOC 2012
VOC_ROOT = "/kaggle/input/datasets/gopalbhattrai/pascal-voc-2012-dataset/VOC2012_test/VOC2012_test"
OUTPUTS_DIR = "/kaggle/working/outputs"

if os.path.exists(VOC_ROOT):
    print("🚀 Bắt đầu đánh giá chéo trên Pascal VOC 2012...")
    !python /kaggle/working/ECAAL/src/cross_evaluate.py --voc-root {VOC_ROOT} --outputs-dir {OUTPUTS_DIR}
    
    print("\n📦 Đang nén kết quả (loại trừ file .pth để tránh quá nặng)...")
    !zip -q -r ecaal_final_results.zip outputs -x "*.pth"
    print("✅ Hoàn tất! File kết quả được lưu tại: /kaggle/working/ecaal_final_results.zip")
else:
    print(f"❌ Không tìm thấy dataset VOC 2012 tại {VOC_ROOT}")
    print("Vui lòng Add Data: gopalbhattrai/pascal-voc-2012-dataset trước khi chạy!")


## Cell 17: COCO In-Distribution Test Evaluation
Đánh giá 6 model trên tập **val COCO 2017** (in-distribution — cùng phân phối với tập train).

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
from pathlib import Path
import sys

sys.path.insert(0, str(REPO_DIR / 'src'))
from models import build_model
from dataset import COCOMultiLabelDataset, get_val_transform
from evaluate import compute_map, compute_f1

COCO_ROOT   = '/kaggle/input/coco-2017-dataset'
OUTPUTS_DIR = Path('/kaggle/working/outputs')
SUBSET_DIR  = Path('/kaggle/working/data/coco_subset')

# Load val subset IDs dùng trong training
val_ids_path = SUBSET_DIR / 'subset_val_ids.json'
val_ids = json.load(open(val_ids_path)) if val_ids_path.exists() else None
print(f"Val IDs: {len(val_ids) if val_ids else 'full val set'}")

device    = 'cuda'
transform = get_val_transform(img_size=224)
val_ds    = COCOMultiLabelDataset(COCO_ROOT, 'val', transform, val_ids)
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=64, num_workers=2, pin_memory=True)

EXPERIMENTS = [
    ('A: ResNet50+BCE',      'exp_A_resnet_bce',            'resnet50',        False),
    ('B: ResNet50+ASL',      'exp_B_resnet_asl',            'resnet50',        False),
    ('C: EffNet+CBAM+ASL',   'exp_C_efficientnet_cbam_asl', 'efficientnet_b0', True),
    ('D: ResNet50+Focal',    'exp_D_resnet_focal',          'resnet50',        False),
    ('E: ResNet50+CBAM+ASL', 'exp_E_resnet_cbam_asl',       'resnet50',        True),
    ('F: EffNet+ASL',        'exp_F_efficientnet_asl',      'efficientnet_b0', False),
]

coco_test_results = []
for name, exp_dir, backbone, use_cbam in EXPERIMENTS:
    pth_path = OUTPUTS_DIR / exp_dir / 'best.pth'
    if not pth_path.exists():
        print(f'⚠️  {name}: best.pth not found — skip')
        continue
    print(f'\n🔍 Evaluating COCO: {name}')
    try:
        model = build_model({'backbone': backbone, 'use_cbam': use_cbam,
                             'num_classes': 80, 'pretrained': False}).to(device)
        model.load_state_dict(torch.load(pth_path, map_location=device))
        model.eval()
        all_probs, all_targets = [], []
        with torch.no_grad():
            for imgs, targets in val_loader:
                probs = torch.sigmoid(model(imgs.to(device))).cpu().numpy()
                all_probs.append(probs)
                all_targets.append(targets.numpy())
        P = np.concatenate(all_probs);  T = np.concatenate(all_targets)
        m = {**compute_map(T, P), **compute_f1(T, P)}
        coco_test_results.append({
            'Experiment': name, 'exp_dir': exp_dir,
            'COCO_mAP': round(m['mAP'], 4),
            'COCO_Macro_F1': round(m['macro_f1'], 4),
            'COCO_Micro_F1': round(m['micro_f1'], 4),
        })
        print(f"  ✅ COCO mAP = {m['mAP']:.4f} | Macro-F1 = {m['macro_f1']:.4f}")
    except Exception as e:
        print(f'  ❌ Error: {e}')

df_coco = pd.DataFrame([{k: v for k, v in r.items() if k != 'exp_dir'} for r in coco_test_results])
df_coco.to_csv(OUTPUTS_DIR / 'coco_test_evaluation.csv', index=False)
print('\n' + '='*65)
print('COCO IN-DISTRIBUTION TEST RESULTS')
print('='*65)
print(df_coco.to_string(index=False))
print('='*65)
print(f'\n✅ Saved to {OUTPUTS_DIR}/coco_test_evaluation.csv')


## Cell 18: So Sánh VOC vs COCO — Bảng Tổng Hợp
Kết hợp kết quả đánh giá chéo **VOC 2012** (Cell 16) và đánh giá **COCO in-distribution** (Cell 17).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUTPUTS_DIR = Path('/kaggle/working/outputs')

voc_csv  = OUTPUTS_DIR / 'voc_cross_evaluation.csv'
coco_csv = OUTPUTS_DIR / 'coco_test_evaluation.csv'

if not voc_csv.exists() or not coco_csv.exists():
    missing = []
    if not voc_csv.exists():  missing.append('voc_cross_evaluation.csv  (run Cell 16)')
    if not coco_csv.exists(): missing.append('coco_test_evaluation.csv  (run Cell 17)')
    print(f'⚠️  Missing files: {", ".join(missing)}')
else:
    df_voc  = pd.read_csv(voc_csv)   # cols: Experiment, VOC_mAP, VOC_Macro_F1, VOC_Micro_F1
    df_coco = pd.read_csv(coco_csv)  # cols: Experiment, COCO_mAP, COCO_Macro_F1, COCO_Micro_F1

    # cross_evaluate.py dùng exp_dir làm Experiment — map lại nếu cần
    exp_dir_to_name = {
        'exp_A_resnet_bce':            'A: ResNet50+BCE',
        'exp_B_resnet_asl':            'B: ResNet50+ASL',
        'exp_C_efficientnet_cbam_asl': 'C: EffNet+CBAM+ASL',
        'exp_D_resnet_focal':          'D: ResNet50+Focal',
        'exp_E_resnet_cbam_asl':       'E: ResNet50+CBAM+ASL',
        'exp_F_efficientnet_asl':      'F: EffNet+ASL',
    }
    # Chuẩn hoá cột Experiment của VOC CSV nếu chứa exp_dir
    if df_voc['Experiment'].str.startswith('exp_').any():
        df_voc['Experiment'] = df_voc['Experiment'].map(
            lambda x: exp_dir_to_name.get(x, x)
        )

    df_merged = df_coco[['Experiment', 'COCO_mAP', 'COCO_Macro_F1', 'COCO_Micro_F1']].merge(
        df_voc[['Experiment', 'VOC_mAP', 'VOC_Macro_F1', 'VOC_Micro_F1']],
        on='Experiment', how='outer'
    ).sort_values('COCO_mAP', ascending=False).reset_index(drop=True)

    print('\n' + '='*80)
    print('CROSS-DATASET COMPARISON: COCO (In-Dist) vs VOC (Cross-Dataset)')
    print('='*80)
    print(df_merged.to_string(index=False))
    print('='*80)
    df_merged.to_csv(OUTPUTS_DIR / 'cross_dataset_comparison.csv', index=False)

    # ── Bar chart ──────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(13, 5))
    x = range(len(df_merged))
    w = 0.35
    bars1 = ax.bar([i - w/2 for i in x], df_merged['COCO_mAP'].fillna(0), w,
                   label='COCO mAP (in-dist)', color='#3498db', alpha=0.88)
    bars2 = ax.bar([i + w/2 for i in x], df_merged['VOC_mAP'].fillna(0), w,
                   label='VOC mAP (cross-dataset)', color='#e67e22', alpha=0.88)
    ax.set_xticks(list(x))
    ax.set_xticklabels(df_merged['Experiment'], rotation=15, ha='right', fontsize=9)
    ax.set_ylabel('mAP', fontsize=11)
    ax.set_title('COCO vs VOC mAP — Ablation Study (6 Models)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    for bar in list(bars1) + list(bars2):
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.004,
                    f'{h:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(OUTPUTS_DIR / 'cross_dataset_bar.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\n✅ Saved: cross_dataset_comparison.csv & cross_dataset_bar.png')


## Cell 19: Train vs Val — Phân Tích Overfitting
Đánh giá lại mỗi model trên **4 000 ảnh COCO train** để so sánh Train mAP vs Val mAP.

- **Gap = Train mAP − Val mAP** : gap < 0.04 → tốt | 0.04–0.08 → nhẹ | > 0.08 → overfit

In [ ]:
import torch, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

sys.path.insert(0, str(REPO_DIR / 'src'))
from models import build_model
from dataset import COCOMultiLabelDataset, get_val_transform
from evaluate import compute_map, compute_f1

COCO_ROOT      = '/kaggle/input/coco-2017-dataset'
OUTPUTS_DIR    = Path('/kaggle/working/outputs')
SUBSET_DIR     = Path('/kaggle/working/data/coco_subset')
DEVICE         = 'cuda'
N_TRAIN_SAMPLE = 4000   # sample size = val size

transform = get_val_transform(img_size=224)  # NO augmentation khi eval train

train_ids_path = SUBSET_DIR / 'subset_train_ids.json'
train_ids = json.load(open(train_ids_path)) if train_ids_path.exists() else None

train_ds = COCOMultiLabelDataset(COCO_ROOT, 'train', transform, train_ids)
random.seed(42)
if len(train_ds.samples) > N_TRAIN_SAMPLE:
    train_ds.samples = random.sample(train_ds.samples, N_TRAIN_SAMPLE)
print(f'[Train eval] Using {len(train_ds.samples)} samples (sampled for speed)')
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, num_workers=2, pin_memory=True)

EXPERIMENTS = [
    ('A: ResNet50+BCE',      'exp_A_resnet_bce',            'resnet50',        False),
    ('B: ResNet50+ASL',      'exp_B_resnet_asl',            'resnet50',        False),
    ('C: EffNet+CBAM+ASL',   'exp_C_efficientnet_cbam_asl', 'efficientnet_b0', True),
    ('D: ResNet50+Focal',    'exp_D_resnet_focal',          'resnet50',        False),
    ('E: ResNet50+CBAM+ASL', 'exp_E_resnet_cbam_asl',       'resnet50',        True),
    ('F: EffNet+ASL',        'exp_F_efficientnet_asl',      'efficientnet_b0', False),
]

overfit_rows = []
for name, exp_dir, backbone, use_cbam in EXPERIMENTS:
    pth_path = OUTPUTS_DIR / exp_dir / 'best.pth'
    log_path = OUTPUTS_DIR / exp_dir / 'log.json'
    if not pth_path.exists():
        print(f'⚠️  {name}: best.pth not found — skip')
        continue
    print(f'\n🔍 Train-eval: {name}')
    try:
        model = build_model({'backbone': backbone, 'use_cbam': use_cbam,
                             'num_classes': 80, 'pretrained': False}).to(DEVICE)
        model.load_state_dict(torch.load(pth_path, map_location=DEVICE))
        model.eval()
        all_probs, all_targets = [], []
        with torch.no_grad():
            for imgs, targets in train_loader:
                probs = torch.sigmoid(model(imgs.to(DEVICE))).cpu().numpy()
                all_probs.append(probs)
                all_targets.append(targets.numpy())
        P = np.concatenate(all_probs);  T = np.concatenate(all_targets)
        tr = {**compute_map(T, P), **compute_f1(T, P)}
        train_map = round(tr['mAP'], 4)
        train_f1  = round(tr['macro_f1'], 4)

        # Best val metrics từ log.json
        val_map, val_f1 = None, None
        if log_path.exists():
            records = json.load(open(log_path))
            best = max(records, key=lambda r: r.get('mAP', 0))
            val_map = round(best.get('mAP', 0), 4)
            val_f1  = round(best.get('macro_f1', 0), 4)

        gap = round(train_map - val_map, 4) if val_map is not None else None
        label = '🟢 OK' if gap is not None and abs(gap) < 0.04 else                 ('🟡 Nhẹ' if gap is not None and abs(gap) < 0.08 else '🔴 Overfit')
        print(f'  Train mAP={train_map:.4f} | Val mAP={val_map:.4f} | Gap={gap:+.4f}  {label}')
        overfit_rows.append({
            'Experiment': name,
            'Train_mAP': train_map,
            'Val_mAP':   val_map,
            'Gap':       gap,
            'Train_F1':  train_f1,
            'Val_F1':    val_f1,
        })
    except Exception as e:
        print(f'  ❌ Error: {e}')

if not overfit_rows:
    print('\n⚠️  No results — run training cells first.')
else:
    df_ov = pd.DataFrame(overfit_rows).sort_values('Val_mAP', ascending=False).reset_index(drop=True)

    print('\n' + '='*75)
    print('OVERFITTING ANALYSIS — Train vs Val mAP (best epoch)')
    print('='*75)
    print(df_ov[['Experiment','Train_mAP','Val_mAP','Gap','Train_F1','Val_F1']].to_string(index=False))
    print('='*75)
    df_ov.to_csv(OUTPUTS_DIR / 'overfitting_analysis.csv', index=False)

    # ── Grouped bar + gap line ──────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    exps = df_ov['Experiment']
    x, w = range(len(exps)), 0.35

    for ax, col_tr, col_val, ylabel, title in [
        (axes[0], 'Train_mAP', 'Val_mAP', 'mAP',      'mAP: Train vs Val (best epoch)'),
        (axes[1], 'Train_F1',  'Val_F1',  'Macro F1', 'Macro F1: Train vs Val (best epoch)'),
    ]:
        b1 = ax.bar([i - w/2 for i in x], df_ov[col_tr].fillna(0), w,
                    label='Train', color='#2ecc71', alpha=0.88)
        b2 = ax.bar([i + w/2 for i in x], df_ov[col_val].fillna(0), w,
                    label='Val',   color='#e74c3c', alpha=0.88)
        ax.set_xticks(list(x))
        ax.set_xticklabels(exps, rotation=15, ha='right', fontsize=8.5)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        for bar in list(b1) + list(b2):
            h = bar.get_height()
            if h > 0:
                ax.text(bar.get_x() + bar.get_width()/2, h + 0.003,
                        f'{h:.3f}', ha='center', va='bottom', fontsize=7.5)

    # Gap overlay trên subplot mAP
    gaps = df_ov['Gap'].fillna(0)
    ax_gap = axes[0].twinx()
    ax_gap.plot(list(x), gaps, 'o--', color='#8e44ad', linewidth=2,
                markersize=7, label='Gap (Train−Val)', zorder=5)
    ax_gap.axhline(0.04,  color='#f39c12', linewidth=1, linestyle=':', alpha=0.7)
    ax_gap.axhline(0.08,  color='#e74c3c', linewidth=1, linestyle=':', alpha=0.7)
    ax_gap.axhline(0,     color='gray',    linewidth=0.8, linestyle=':')
    ax_gap.set_ylabel('Gap (Train − Val)', color='#8e44ad', fontsize=10)
    ax_gap.tick_params(axis='y', labelcolor='#8e44ad')
    ax_gap.legend(loc='upper right', fontsize=9)

    plt.suptitle('Overfitting Analysis — 6 Models on COCO Subset', fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(str(OUTPUTS_DIR / 'overfitting_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\n✅ Saved: overfitting_analysis.csv & overfitting_analysis.png')
